In [211]:
import pandas as pd
import re
import numpy as np
df = pd.read_csv("naukari.csv")

## Salary Processing Pipeline

1. Read `naukari.csv` into `df`.
2. Define converters:
  - `if not disclosed`
    - then set to "Not Disclosed-Not Disclosed"
  - `convert_dollar_to_lacs(salary)`:
    - strip "$", commas
    - extract numeric values
    - multiply by 80 (INR conversion)
    - format as `low-high` or single value
  - `convert_cr_to_lacs(salary)`:
    - strip "₹"
    - extract numeric values
    - multiply by 1e7
    - format as range or single value
  - `convert_salary_value(x)`:
    - if contains `"Cr"`, call `convert_cr_to_lacs`
    - elif contains `"$"`, call `convert_dollar_to_lacs`
    - else clean up
     - strip commas, quotes, `₹`, `P.A.`, `Lacs`
     - parse hyphen range
     - if numeric and <500, multiply by 100000 (to lacs→annual INR)
     - return `"low-high"` (or same for fixed)
    - non-string: `"Not Disclosed"`

3. Apply to `df["salary"]`:
  - `df["salary"] = df["salary"].apply(convert_salary_value)`

4. Export cleaned values:
  - `df["salary"].to_csv("cleaned_salaries.csv", index=False)`

In [212]:
#convert salary values to lacs
def convert_dollar_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("$", "").strip()
  salary = salary.replace(",", "")
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 80 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  if result== "":
    print(salary)
  return result

#convert salary values in crores to lacs
def convert_cr_to_lacs(salary):
  if pd.isna(salary) or not isinstance(salary, str):
    return salary
  salary = salary.replace("\u20B9", "").strip()
  is_above = "and above" in salary
  numbers = re.findall(r"\d+\.?\d*", salary)
  numbers_lacs = [float(num) * 10000000 for num in numbers]
  if len(numbers_lacs) == 2:
    result = f"{numbers_lacs[0]}-{numbers_lacs[1]}"
  elif len(numbers_lacs) == 1:
    result = f"{numbers_lacs[0]}"
  else:
    result = salary
  return result

def convert_salary_value(x):
    if isinstance(x, str) and "Not Disclosed" in x:
        return "Not Disclosed-Not Disclosed"
    if isinstance(x, str) and "Cr" in x:
        return convert_cr_to_lacs(x)
    elif isinstance(x, str) and "$" in x:
        return convert_dollar_to_lacs(x)
    elif isinstance(x, str):
        cleanrange = x.replace(',', '').strip().replace('"', '').strip().replace("₹", "").strip().replace("P.A.", "").strip().replace("Lacs", "").strip().split('(')[0].strip()
        if cleanrange.find('-') != -1:
            parts = cleanrange.split('-')
            if len(parts) == 2:
                try:
                  low = float(parts[0])
                  high = float(parts[1])
                  if low < 500:
                    low *= 100000
                  if high < 500:
                    high *= 100000
                  cleanrange = f"{low}-{high}"
                except ValueError:
                  cleanrange = f"{parts[0]}-{parts[1]}"
            elif len(parts) == 1:
              cleanrange = f"{parts[0]}"
        else:
            try:
              value = float(cleanrange)
              if value < 500:
                value *= 100000
              cleanrange = f"{value}-{value}"
            except ValueError:
              cleanrange = f"{cleanrange}"
        return cleanrange
    else:
        return "Not Disclosed-Not Disclosed"

df["salary"] = df["salary"].apply(convert_salary_value).astype(str)
df[["min_salary", "max_salary"]] = df["salary"].str.split("-", expand=True)



`exp_req` is normalized by `clean_experience`:

- if missing/NaN or not string: kept as-is.
- if `"not disclosed"`/`"not specified"`/`"not mentioned"` (case-insensitive): converted to `"Not Disclosed"`.
- otherwise extract all numeric tokens using regex `\d+\.?\d*`.
- if two numbers found: set `"min-max"` (e.g. `3-5` stays `3-5`).
- if one number found: duplicate as range `"x-x"` (e.g. `4` -> `4-4`).
- if none, leave original text.

Then:
- `df["exp_req"]` set to string type.
- split into two columns:
  - `min_exp` (before `-`)
  - `max_exp` (after `-`)

In [213]:
def clean_experience(x):
    if pd.isna(x) or not isinstance(x, str):
        return x
    x = x.strip()
    if x.lower() in ["not disclosed", "not specified", "not mentioned"]:
        return "Not Disclosed"
    numbers = re.findall(r"\d+\.?\d*", x)
    if len(numbers) == 2:
        return f"{numbers[0]}-{numbers[1]}"
    elif len(numbers) == 1:
        return f"{numbers[0]}-{numbers[0]}"
    else:
        return x

df["exp_req"] = df["exp_req"].apply(clean_experience).astype(str)
df[["min_exp", "max_exp"]] = df["exp_req"].str.split("-", expand=True)


In [214]:
df["working_time"] = df["emp_type"].str.split(',').str[0].str.strip()
df["contract_type"] = df["emp_type"].str.split(',').str[1].str.strip()
df.head(100)


,Unnamed: 0,job_title,company,rating,exp_req,salary,location,post_date,applicants,role,...,emp_type,role_category,UG,key_skills,min_salary,max_salary,min_exp,max_exp,working_time,contract_type
0,0,Probationary Officers' Programme,Icici Bank,4.0,0-5,425000.0-450000.0,"['Kolkata,', 'Hyderabad/Secunderabad,', 'Pune,...",Posted: Few Hours Ago,Job Applicants: 88751,Retail Banking Sales,...,"Full Time, Permanent",Retail & B2C Sales,Any Graduate,"['Graduate', 'Customer Service', 'Sales', 'Bfs...",425000.0,450000.0,0,5,Full Time,Permanent
1,1,Editorial Reviewer: Medicine and Life Sciences...,CACTUS Communications,3.5,0-4,Not Disclosed-Not Disclosed,"['Kolkata,', 'Pune,', 'Chennai,', 'Bangalore/B...",Posted: 1 day ago,Job Applicants: 88751,Editor / Content Analyst,...,"Full Time, Temporary/Contractual",Editing (Print / Online / Electronic),"BDS in Any Specialization, B.Pharma in Any Spe...","['editing', 'Medicine', 'Biotechnology', 'Cont...",Not Disclosed,Not Disclosed,0,4,Full Time,Temporary/Contractual
2,2,Job Opening | Chat Process | Day Shift | MNC,Trigent Software,3.7,0-0,Not Disclosed-Not Disclosed,"['Kolkata,', 'Hyderabad/Secunderabad,', 'Pune,...",Posted: 1 day ago,Job Applicants: 687,Email Support,...,"Full Time, Permanent",Non Voice,Any Graduate,"['Non Voice', 'Chat Process']",Not Disclosed,Not Disclosed,0,0,Full Time,Permanent
3,3,Customer Support Executive,Careernet Technologies,NaN,0-5,250000.0-475000.0,"['Bangalore/Bengaluru,', 'Mumbai', '(All', 'Ar...",Posted: 1 day ago,Job Applicants: Less than 10,Customer Retention - Voice / Blended,...,"Full Time, Permanent",Voice / Blended,Graduation Not Required,"['international voice process', 'International...",250000.0,475000.0,0,5,Full Time,Permanent
4,4,Top most International Bpo Hiring with the bes...,Careernet Technologies,4.2,0-3,250000.0-400000.0,"['Bangalore/Bengaluru,', 'Mumbai', '(All', 'Ar...",Posted: 2 days ago,Job Applicants: 17,Customer Service,...,"Full Time, Permanent","Customer Success, Service & Operations - Other",Any Graduate,"['international voice process', 'US Process', ...",250000.0,400000.0,0,3,Full Time,Permanent
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,102,ASSOCIATE TRAINEE- SALES & SERVICE,Malabar Group.,4.0,0-1,Not Disclosed-Not Disclosed,"['Mumbai,', 'Visakhapatnam,', 'New', 'Delhi,',...",Posted: 30+ days ago,Job Applicants: 9349,Direct Sales Executive,...,"Full Time, Permanent",Retail & B2C Sales,Any Graduate,"['Sales Staff', 'B2B', 'Sales Representative',...",Not Disclosed,Not Disclosed,0,1,Full Time,Permanent
96,103,Voice Process,Cognizant,3.9,0-4,Not Disclosed-Not Disclosed,"['Hyderabad/Secunderabad,', 'Pune,', 'Bangalor...",Posted: 18 days ago,Job Applicants: 1227,Voice / Blended - Other,...,"Full Time, Permanent",Voice / Blended,Any Graduate,"['international voice process', 'healthcare', ...",Not Disclosed,Not Disclosed,0,4,Full Time,Permanent
97,106,UI Developer,Gupshup Technology,4.3,0-3,Not Disclosed-Not Disclosed,"['Mumbai,', 'New', 'Delhi,', 'Bangalore/Bengal...",Posted: 30+ days ago,NaN,Front End Developer,...,"Full Time, Permanent",Software Development,Any Graduate,"['Usage', 'CSS', 'User interface designing', '...",Not Disclosed,Not Disclosed,0,3,Full Time,Permanent
98,107,International Voice Process /Pune/ Bangalore /...,Cognizant,3.9,0-4,Not Disclosed-Not Disclosed,"['Hyderabad/Secunderabad,', 'Pune,', 'Bangalor...",Posted: 30+ days ago,Job Applicants: 4865,Voice / Blended - Other,...,"Full Time, Permanent",Voice / Blended,Any Graduate,"['international voice process', 'customer supp...",Not Disclosed,Not Disclosed,0,4,Full Time,Permanent
